# Analiza Sieci Gier Steam - Out-Of-Core (Neo4j)
Zgodnie z naszym planem, przenieśliśmy logikę do bazy grafowej Neo4j. Dzięki temu unikamy limitów pamięci RAM w Pythonie. Baza przechowuje pełen graf (8.3 GB) na dysku twardym i wykorzystuje silnik **Graph Data Science (GDS)** do obliczeń w wysokiej wydajności.

W tym notatniku łączymy się z bazą lokalnie, zlecamy wczytanie plików z folderu `/data` za pomocą wbudowanych mechanizmów szybkiego ładowania (LOAD CSV), a następnie uruchamiamy algorytmy topologiczne.


jaccard część wspólna zbiorów licznoś c iloczynu przez licznośc sumy, bez 0 playtimeu, cosinusow jaccard z pierwiastkowanym mianownikiem - łądgodniejszy jaccard, overlap jaki procent graczy mniejszej gry posiada rownież tą wiekszą, alokacja liczba krawęðż 1/liczbe gier danego gracza

In [2]:
from neo4j import GraphDatabase
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

# Ustawienie wyświetlania
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 10)
sns.set_theme(style="whitegrid")

# Dane dostępowe (domyślne z docker-compose)
URI = "bolt://neo4j:7687"
# UWAGA: Jeśli uruchamiasz to WEWNĄTRZ kontenera jupyter przez docker-compose, użyj:
# URI = "bolt://neo4j:7687"

AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)

def run_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters)
        return pd.DataFrame([r.values() for r in result], columns=result.keys())

# Test połączenia
try:
    driver.verify_connectivity()
    print("Połączono pomyślnie z Neo4j!")
except Exception as e:
    print(f"Błąd połączenia: {e}")
    print("Upewnij się, że 'docker-compose up -d' zostało uruchomione i baza wstała.")


Połączono pomyślnie z Neo4j!


## Połączenie i Schemat

In [3]:
# Krok 1: Inicjalizacja schematu bazy danych
# Najpierw tworzymy index/constraint na polu `id`, żeby łączenie krawędzi podczas wczytywania 8GB działało błyskawicznie (O(1)).

constraint_query = '''
CREATE CONSTRAINT game_id IF NOT EXISTS FOR (g:Game) REQUIRE g.id IS UNIQUE;
'''

try:
    with driver.session() as session:
        session.run(constraint_query)
    print("Utworzono indeks na Game(id).")
except Exception as e:
    print(f"Informacja: {e}")


Utworzono indeks na Game(id).


## Szybkie Ładowanie (LOAD CSV) do bazy na dysku

In [4]:
# Krok 2: Wczytanie węzłów (Game Metadata) z pliku
# Neo4j z docker-compose automatycznie widzi folder ./data jako file:///

load_nodes_query = '''
LOAD CSV WITH HEADERS FROM 'file:///game_metadata.csv' AS row
MERGE (g:Game {id: row.id})
SET g.name = row.name, 
    g.developer = row.developer, 
    g.genres = row.genres;
'''

print("Wczytywanie gier...")
with driver.session() as session:
    summary = session.run(load_nodes_query).consume()
    print(f"Utworzono węzłów: {summary.counters.nodes_created}")


Wczytywanie gier...
Utworzono węzłów: 44023


In [ ]:
# Krok 3: Wczytanie 8.3GB Krawędzi w transakcjach
# Używamy mechanizmu CALL { ... } IN TRANSACTIONS, który zapobiega przepełnieniu RAMu.
# To może potrwać od kilku do kilkunastu minut w zależności od prędkości dysku.

load_edges_query = '''
LOAD CSV WITH HEADERS FROM 'file:///game_game_rich_projection.csv' AS row
CALL {
  WITH row
  MATCH (s:Game {id: row.source})
  MATCH (t:Game {id: row.target})
  MERGE (s)-[r:SHARED_PLAYERS]->(t)
  SET r.shared_players = toInteger(row.shared_players),
      r.jaccard = toFloat(row.jaccard)
} IN TRANSACTIONS OF 100000 ROWS;
'''

print("Wczytywanie 8.3GB krawędzi do Neo4j w paczkach po 100k... (proszę o cierpliwość)")
with driver.session() as session:
    summary = session.run(load_edges_query).consume()
    print(f"Dodano relacji: {summary.counters.relationships_created}")


Wczytywanie 8.3GB krawędzi do Neo4j w paczkach po 100k... (proszę o cierpliwość)


## Algorytmy na dysku (Neo4j Graph Data Science)

In [ ]:
# Krok 4: Projektowanie grafu do pamięci GDS (Graph Data Science)
# GDS w Neo4j operuje na specjalnej zoptymalizowanej strukturze IN-MEMORY dla algorytmów.

project_query = '''
CALL gds.graph.project(
  'steamGraph',
  'Game',
  {
    SHARED_PLAYERS: {
      orientation: 'UNDIRECTED',
      properties: ['jaccard', 'shared_players']
    }
  }
)
YIELD graphName, nodeCount, relationshipCount, createMillis;
'''

try:
    res = run_query(project_query)
    display(res)
except Exception as e:
    # Jeśli projekt już istnieje
    print("Graf 'steamGraph' został już załadowany do GDS. Aby go usunąć i wgrać ponownie odpal: CALL gds.graph.drop('steamGraph')")


In [ ]:
# Krok 5: Analiza Centralności Wierzchołka (Stopień)
# Robimy to bazowym zapytaniem w Neo4j, szybkim i zoptymalizowanym.

degree_query = '''
MATCH (g:Game)-[r:SHARED_PLAYERS]-()
WITH g, count(r) as degree
ORDER BY degree DESC
LIMIT 10
RETURN g.name AS Game, degree AS Degree
'''

print("Top 10 Gier wg Stopnia (Degree Centrality):")
display(run_query(degree_query))


In [ ]:
# Krok 6: PageRank z wagą 'jaccard' przez GDS
pagerank_query = '''
CALL gds.pageRank.stream('steamGraph', {
  relationshipWeightProperty: 'jaccard'
})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS Game, score AS PageRank_Jaccard
ORDER BY score DESC LIMIT 10
'''

print("Top 10 Gier wg PageRank (Jaccard):")
display(run_query(pagerank_query))


## Grupowanie i Wizualizacja

In [ ]:
# Krok 7: Wykrywanie społeczności (Louvain) przez GDS
# Ten algorytm zapisuje wyniki (identyfikatory klastrów) z powrotem do naszej grafowej bazy (mutate/write).
# Zapiszemy to jako właściwość węzła `community`.

louvain_query = '''
CALL gds.louvain.write('steamGraph', {
  relationshipWeightProperty: 'jaccard',
  writeProperty: 'community'
})
YIELD communityCount, modularity, computeMillis
RETURN communityCount, modularity, computeMillis
'''

print("Uruchamianie algorytmu Louvain...")
display(run_query(louvain_query))

# Sprawdzenie jakie gry trafiły do jakich społeczności (wybieramy kilka popularnych)
sample_community_query = '''
MATCH (g:Game)
WHERE g.community IS NOT NULL
RETURN g.community AS CommunityID, count(g) AS Size, collect(g.name)[0..3] AS ExampleGames
ORDER BY Size DESC LIMIT 5
'''
print("\nNajwiększe społeczności:")
display(run_query(sample_community_query))


In [ ]:
# Krok 8: Pobranie subgrafu do RAM uzywając NetworkX w celu wizualizacji rdzenia (Hubów)
# Zamiast pobierać 8GB, po prostu każemy bazie zwrócić ścieżki między 50 najważniejszymi węzłami.

subgraph_query = '''
MATCH (g1:Game)-[r:SHARED_PLAYERS]-(g2:Game)
// chcemy tylko te najważniejsze
WHERE g1.community IS NOT NULL AND g2.community IS NOT NULL
// ograniczymy do powiedzmy losowych 200 silnych krawędzi w największej spolecznosci, żeby coś ładnie narysować
WITH g1, g2, r
ORDER BY r.jaccard DESC LIMIT 500
RETURN g1.id AS source, g1.name AS source_name, g1.community AS source_comm,
       g2.id AS target, g2.name AS target_name, g2.community AS target_comm,
       r.jaccard AS weight
'''

df_subgraph = run_query(subgraph_query)

# Budujemy mały graf z wyników i wizualizujemy
G = nx.Graph()
for idx, row in df_subgraph.iterrows():
    G.add_node(row['source'], name=row['source_name'], community=row['source_comm'])
    G.add_node(row['target'], name=row['target_name'], community=row['target_comm'])
    G.add_edge(row['source'], row['target'], weight=row['weight'])

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.5, weight='weight', seed=42)

# Kolorowanie po community
communities = [G.nodes[n]['community'] for n in G.nodes()]
node_sizes = [G.degree(n) * 50 for n in G.nodes()]
labels = {n: G.nodes[n]['name'] for n in G.nodes()}

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=communities, cmap=plt.cm.Set3, alpha=0.9)
nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray')
nx.draw_networkx_labels(G, pos, labels=labels, font_size=9)

plt.title("Zwizualizowany rdzeń największych powiązań (z klastrami Louvain)")
plt.axis('off')
plt.show()

driver.close()
